# Eigenvalues and Diagonalization

We now explore how matrices stretch along eigenvectors and use power iteration to approximate the dominant eigenpair, comparing against NumPy's eigensolver.


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from linalg_utils.eigen import eigen_2x2, power_iteration, rayleigh_quotient

FIGURES_DIR = Path.cwd() / "assets" / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


## Stretching along eigenvectors

A matrix elongates space along its eigenvectors. We compute the analytical eigenpairs in 2×2 before plotting how the unit circle becomes an ellipse aligned with those directions.


In [ ]:
A = np.array([[3.0, 1.0], [1.0, 2.0]], dtype=np.float64)
res = eigen_2x2(A)
print(f"Eigenvalues: {res.eigenvalues}")
print("Eigenvectors (columns):\n", res.eigenvectors)

theta = np.linspace(0.0, 2 * np.pi, 200)
circle = np.column_stack([np.cos(theta), np.sin(theta)])
transformed = (A @ circle.T).T

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(circle[:, 0], circle[:, 1], linestyle=":", color="gray", label="unit circle")
ax.plot(transformed[:, 0], transformed[:, 1], color="tab:blue", label="A @ circle")
for idx, vec in enumerate(res.eigenvectors.T):
    ax.plot([0, vec[0]], [0, vec[1]], linewidth=3, label=f"eigen {idx + 1}")
ax.set_aspect("equal", adjustable="box")
ax.set_title("Eigenvectors stretch the unit circle")
ax.grid(True, linestyle=":")
ax.legend()
path = FIGURES_DIR / "eigen_stretch.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved eigen stretch figure to {path}")


## Power iteration and eigenvalue convergence

We apply power iteration to a symmetric matrix, track the Rayleigh quotient, and compare the result with NumPy's solver.


In [ ]:
matrix = np.array([[4.0, 1.5], [1.5, 3.0]], dtype=np.float64)
result = power_iteration(matrix, tol=1e-12)
print(f"Power iteration: eigenvalue={result.eigenvalue:.6f} after {result.iterations} steps")
values, vectors = np.linalg.eig(matrix)
print(f"NumPy max eigenvalue: {np.max(values):.6f}")
print(f"Rayleigh quotient: {rayleigh_quotient(matrix, result.eigenvector):.6f}")

history = []
x = np.asarray(result.eigenvector, dtype=np.float64)
for _ in range(30):
    y = matrix @ x
    x = y / np.linalg.norm(y)
    history.append(rayleigh_quotient(matrix, x))

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(history, marker="o")
ax.hlines(np.max(values), 0, len(history) - 1, colors="tab:orange", linestyles="--", label="NumPy max eig")
ax.set_xlabel("Iteration")
ax.set_ylabel("Rayleigh quotient")
ax.set_title("Power iteration converges to the dominant eigenvalue")
ax.grid(True, linestyle=":")
ax.legend()
path = FIGURES_DIR / "power_iteration_convergence.png"
fig.savefig(path, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved power iteration figure to {path}")


In [ ]:
from linalg_utils.checks import assert_close
from linalg_utils.eigen import power_iteration
import numpy as np
matrix = np.array([[4.0, 1.5], [1.5, 3.0]], dtype=np.float64)
result = power_iteration(matrix, tol=1e-12)
values, _ = np.linalg.eig(matrix)
assert_close('dominant eigenvalue', result.eigenvalue, np.max(values), tol=1e-6)
print('Rayleigh quotient matches NumPy dominant eigenvalue within tolerance.')